# Gradient Boosting (Pipeline B)

**Model**: `GradientBoostingRegressor` / **Target**: gdpc1
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 6
**Scaling**: NO / **HP tuning**: YES / **10-model averaging**: YES
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET)

SEED=42; np.random.seed(SEED)
TRAIN_START='1959-01-01'; TRAIN_END='2007-12-31'; VAL_END='2016-12-31'
TEST_START='2017-01-01'; TEST_END='2025-12-31'
N_LAGS=4; N_MODELS=10
VINTAGES={'m1':-2,'m2':-1,'m3':0}
print('GB â€” Cat3, n_lags=4, 10-model avg')

GB â€” Cat3, n_lags=4, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print(f'Features: {len(features)}')


Features: 56


In [3]:
# Phase A: HP tuning on train+val (1959-2016) via TimeSeriesSplit
print('Phase A: HP tuning on train+val (1959-2016)...')

tune_data = monthly[(monthly['date']>=TRAIN_START)&(monthly['date']<=VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how='any').reset_index(drop=True)
feature_cols = [c for c in tune_flat.columns if c!='date' and c!=TARGET and any(c==f or c.startswith(f+'_') for f in features)]

tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf
best_params = {'learning_rate': 0.1, 'n_estimators': 200, 'max_depth': 3}

for lr in [0.01, 0.05, 0.1]:
    for ne in [200, 500]:
        for md in [3, 5]:
            scores = []
            for tr, va in tscv.split(tune_flat):
                m = GradientBoostingRegressor(loss='squared_error', learning_rate=lr,
                    n_estimators=ne, max_depth=md, max_features='sqrt', random_state=SEED)
                m.fit(tune_flat[feature_cols].values[tr], tune_flat[TARGET].values[tr])
                p = m.predict(tune_flat[feature_cols].values[va])
                scores.append(np.mean((p - tune_flat[TARGET].values[va])**2))
            if np.mean(scores) < best_score:
                best_score = np.mean(scores)
                best_params = {'learning_rate': lr, 'n_estimators': ne, 'max_depth': md}

print(f'Best GB params: {best_params}')


Phase A: HP tuning on train+val (1959-2016)...


Best GB params: {'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 3}


In [4]:
test_quarters = monthly[(monthly['date']>=TEST_START)&(monthly['date']<=TEST_END)&
                         monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v:[] for v in VINTAGES}; actuals_list=[]; failed=0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date']==pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly['date']>=TRAIN_START)&(monthly['date']<=fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how='any').reset_index(drop=True)

        if len(train_fl)<10: predictions[vn].append(np.nan); continue

        feature_cols = [c for c in train_fl.columns if c != 'date' and c != TARGET and any(c == f or c.startswith(f + '_') for f in features)]
        try:
            models=[]
            for k in range(N_MODELS):
                m=GradientBoostingRegressor(loss='squared_error',
                    learning_rate=best_params['learning_rate'],
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    max_features='sqrt',random_state=SEED+k)
                m.fit(train_fl[feature_cols].values,train_fl[TARGET].values)
                models.append(m)
            tr=monthly[monthly['date']==fc].reset_index(drop=True)
            tr_m=gen_lagged_data(metadata,tr.copy(),fc,lag=0)
            tr_f=mean_fill_dataset(train_m,tr_m)
            ctx=train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb=pd.concat([ctx,tr_f],ignore_index=True)
            cmb_fl=flatten_data(cmb,TARGET,N_LAGS)
            tr_fl=cmb_fl.tail(1)
            ctx=train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb=pd.concat([ctx,tr_f],ignore_index=True)
            cmb_fl=flatten_data(cmb,TARGET,N_LAGS)
            tr_fl=cmb_fl.tail(1)
            preds=[m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed+=1

    if (i+1)%6==0: print(f'  {i+1} / {len(test_quarters)}')
print(f'Done. {failed} failures.')

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

  36 / 36
Done. 0 failures.


In [5]:
os.makedirs(PREDICTIONS_DIR,exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date':test_quarters,'actual':actuals_list,'prediction':predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR,f'gb_{vn}.csv'),index=False)
    print(f'Saved gb_{vn}.csv')

Saved gb_m1.csv
Saved gb_m2.csv
Saved gb_m3.csv


In [6]:
def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan
def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan
panels={'Pre-COVID':'2017-01-01,2019-12-31','COVID':'2020-04-01,2021-12-31',
        'Post-COVID':'2022-01-01,2025-12-31','Full':'2017-01-01,2025-12-31'}
a=np.array(actuals_list); d=pd.to_datetime(test_quarters)
for pn,rng in panels.items():
    ps,pe=rng.split(','); m=(d>=ps)&(d<=pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp=np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m],pp[m]):.6f}  MAE={mae(a[m],pp[m]):.6f}')

--- Pre-COVID ---
  m1  RMSFE=0.002814  MAE=0.002224
  m2  RMSFE=0.002770  MAE=0.002157
  m3  RMSFE=0.002665  MAE=0.002090
--- COVID ---
  m1  RMSFE=0.042089  MAE=0.026136
  m2  RMSFE=0.041188  MAE=0.025720
  m3  RMSFE=0.042472  MAE=0.026288
--- Post-COVID ---
  m1  RMSFE=0.004820  MAE=0.003558
  m2  RMSFE=0.004737  MAE=0.003475
  m3  RMSFE=0.004501  MAE=0.003375
--- Full ---
  m1  RMSFE=0.019214  MAE=0.007976
  m2  RMSFE=0.018820  MAE=0.007838
  m3  RMSFE=0.019335  MAE=0.007879
